# QAT to MobileNetV2 and lower & export quantized model to XNNPACK using ExecuTorch

##  Imports

#### Import MobileNetV2 model and dataset from Torchvision

In [1]:
import time
import copy
import torch
import torchao
import torchvision
import torchvision.models as models
from torchvision.models.mobilenetv2 import MobileNet_V2_Weights
from torchvision import datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#### Import Backend-specific Quantizer (XNNPACK), ExecuTorch function to lower the model, and PT2E quantizer functions for Quantization-Aware Training

In [2]:
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_qat_pt2e

## Prepare Dataset

In [3]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
dataset = datasets.CIFAR10('~/data/cifar10', download = True, 
            transform=transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize]))
dataset_test = torchvision.datasets.CIFAR10('~/data/cifar10', train = False, download = True, 
            transform=transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize]))

train_sampler = torch.utils.data.RandomSampler(dataset)
test_sampler = torch.utils.data.SequentialSampler(dataset_test)

data_loader = DataLoader(dataset, batch_size=32,
        sampler=train_sampler)

data_loader_test = DataLoader(dataset_test, batch_size=32,
        sampler=test_sampler)

## Model - Import, Export, Prepare for QAT

### Import MobileNetV2 model

In [4]:
model = models.mobilenetv2.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
sample_inputs = (torch.randn(2, 3, 224, 224), )

### Export model using torch.export

In [5]:
dynamic_shapes = tuple(
  {0: torch.export.Dim("dim")} if i == 0 else None
  for i in range(len(sample_inputs))
)
training_ep = torch.export.export(model, sample_inputs, dynamic_shapes=dynamic_shapes).module()

### Prepare Model for Quantization-Aware Training

In [6]:
qparams = get_symmetric_quantization_config(is_per_channel=True, is_qat=True)
quantizer = XNNPACKQuantizer()
quantizer.set_global(qparams)
prepared_model = prepare_qat_pt2e(training_ep, quantizer)

## Training

### Pre-requisite functions

In [11]:
num_epochs = 10
num_train_batches = 20
num_eval_batches = 20
_ = torch.manual_seed(191009)
optimizer = torch.optim.SGD(prepared_model.parameters(), lr=0.001, momentum=0.9)

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def evaluate(model, criterion, data_loader): #, device):
    torchao.quantization.pt2e.move_exported_model_to_eval(model)
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')
    cnt = 0
    with torch.no_grad():
        for image, target in data_loader:
            # image = image.to(device)
            # target = target.to(device)
            output = model(image)
            loss = criterion(output, target)
            cnt += 1
            acc1, acc5 = accuracy(output, target, topk=(1, 5))
            top1.update(acc1[0], image.size(0))
            top5.update(acc5[0], image.size(0))
    print('')

    return top1, top5

### Training Loop

Note that, before inference, you must first call torchao.quantization.pt2e.move_exported_model_to_eval() to ensure certain ops like dropout behave correctly in the eval graph. Otherwise, we would continue to incorrectly apply dropout in the forward pass during inference.

In [12]:
for count_epochs in range(num_epochs):
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')
    avgloss = AverageMeter('Loss', '1.5f')
    count = 0
    for image, target in data_loader:
        start_time = time.time()
        print('.', end='')
        count+=1
        image, target = image.to("cpu"), target.to("cpu")
        output = prepared_model(image)
        loss = torch.nn.CrossEntropyLoss()(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        top1.update(acc1[0], image.size(0))
        top5.update(acc5[0], image.size(0))
        avgloss.update(loss, image.size(0))
        if count >= num_train_batches:
            print('Loss', avgloss.avg)

            print('Training: * Acc@1 {top1.avg:.3f} Acc@5 {top5.avg:.3f}'
                  .format(top1=top1, top5=top5))

    #print('Full train set:  * Acc@1 {top1.global_avg:.3f} Acc@5 {top5.global_avg:.3f}'.format(top1=top1, top5=top5))

    if (count_epochs + 1) % 2 == 0:
        prepared_model_copy = copy.deepcopy(prepared_model)
        quantized_model = convert_pt2e(prepared_model_copy)
        torchao.quantization.pt2e.move_exported_model_to_eval(quantized_model)
        top1, top5 = evaluate(quantized_model, torch.nn.CrossEntropyLoss(), data_loader_test)
        print('Epoch %d: Evaluation accuracy on %d images, %2.2f' % (count_epochs, num_eval_batches * 32, top1.avg))

## Lower and Export the Quantized Model using ExecuTorch on XNNPACK Backend

In [ ]:
et_program = to_edge_transform_and_lower( # (6)
    torch.export.export(quantized_model, sample_inputs),
    partitioner=[XnnpackPartitioner()],
).to_executorch()

with open("mv2_xnnpack.pte", "wb") as file:
    et_program.write_to_file(file)